# LDA

### Construction of vocubulary (tokens)

Same approach as in graph experiment and frequential analysis

In [ ]:
from utils.ml import data_dir, spark, tokenize

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/04/21 15:58:52 WARN Utils: Your hostname, DESKTOP-UQF5BSK, resolves to a loopback address: 127.0.1.1; using 172.24.225.163 instead (on interface eth0)
26/04/21 15:58:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/21 15:58:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df = spark.read.csv(data_dir, header=True, inferSchema=True, sep = ";")
df, tokens = tokenize(df)

TOKENIZED DF:


+----+-----+---+-----+-------+----------+--------------------+
|year|month|day|order|country|session ID|            sequence|
+----+-----+---+-----+-------+----------+--------------------+
|2008|    4|  1|    9|     29|         1|[197, 149, 51, 91...|
|2008|    4|  1|   10|     29|         2|[87, 12, 215, 9, ...|
+----+-----+---+-----+-------+----------+--------------------+
only showing top 2 rows
TOKEN INFO: 
+----------------------+-----------------------+------+--------+-----------------+-----+-------+----+---+
|page 1 (main category)|page 2 (clothing model)|colour|location|model photography|price|price 2|page| id|
+----------------------+-----------------------+------+--------+-----------------+-----+-------+----+---+
|                     4|                    P51|     9|       5|                1|   28|      2|   3|  0|
|                     4|                    P57|     4|       1|                1|   43|      1|   4|  1|
+----------------------+-----------------------+------+

In [ ]:
from pyspark.ml.feature import CountVectorizer
from pyspark.ml.clustering import LDA
from pyspark.ml import Pipeline
num_features = tokens.count()
count_vectorizer = CountVectorizer(vocabSize=num_features, inputCol="sequence", outputCol="tf")
tf = count_vectorizer.fit(df)
tf = tf.transform(df)
tf.show(5)

+----+-----+---+-----+-------+----------+--------------------+--------------------+
|year|month|day|order|country|session ID|            sequence|                  tf|
+----+-----+---+-----+-------+----------+--------------------+--------------------+
|2008|    4|  1|    9|     29|         1|[197, 149, 51, 91...|(218,[0,19,28,34,...|
|2008|    4|  1|   10|     29|         2|[87, 12, 215, 9, ...|(218,[3,8,14,31,5...|
|2008|    4|  1|    6|     21|         3|[91, 54, 151, 142...|(218,[25,28,44,61...|
|2008|    4|  1|    4|     21|         4| [153, 183, 54, 102]|(218,[61,78,111,1...|
|2008|    4|  1|    1|      9|         5|               [128]|    (218,[73],[1.0])|
+----+-----+---+-----+-------+----------+--------------------+--------------------+
only showing top 5 rows


In [4]:
scores = []
best_i = 0
first = True
for i, k in enumerate(range(2, 15)):
    lda = LDA(featuresCol="tf", maxIter=30, k = k, seed = 42)
    model = lda.fit(tf)
    perplexity = model.logPerplexity(tf)
    ll = model.logLikelihood(tf)
    scores.append({"k": k, "lp": perplexity, "ll": ll})
    print("P:", perplexity, "LL:", ll)
    if first:
        first = False
        continue
    if scores[best_i]["ll"] < ll and scores[best_i]["lp"] > perplexity:
        print("Best K:", k)
        best_i = i

P: 5.060513877181475 LL: -837383.4733127274


P: 4.991435817593027 LL: -825952.8504803885
Best K: 3


P: 4.918988156375006 LL: -813964.6461879978
Best K: 4


P: 4.952143186034467 LL: -819450.9415658675


P: 4.9297179438983605 LL: -815740.1470486373


...

P: 4.967027700955277 LL: -821913.9417878735


P: 4.996550169960799 LL: -826799.1428240932


P: 4.9998054635381255 LL: -827337.8092735078


P: 5.061651804289329 LL: -837571.7706629724


P: 5.075530460577247 LL: -839868.3274335593


In [5]:
pipeline = Pipeline(stages = [count_vectorizer, LDA(featuresCol="tf", maxIter=30, k = scores[best_i]["k"], seed = 42)])
fitted = pipeline.fit(df)
fitted.write().overwrite().save("LDA_pipeline")